In [1]:
import os, sys
os.chdir('..')
sys.path.append(os.getcwd())
print('Working directory:', os.getcwd())

Working directory: /Users/lr/policy_vector/shareable


# Mean-Difference Vector Evaluation

Load the pre-generated dataset and mean-difference vector, then evaluate
projection separation on a limited subset (change `limit` to run the full set).

In [2]:
import json
from pathlib import Path
from policy_vector_pipeline import load_dataset, MeanDifferenceVector, ActivationCollector
from scripts.evaluate_vector import compute_stats, load_model
from transformers import logging
logging.set_verbosity_error()

dataset_path = Path('data/on_policy_persona.json')
vector_path = Path('artifacts/qwen3_onpolicy_mean.pt')
model_name = 'Qwen/Qwen3-4B'

dataset = load_dataset(dataset_path)
vector = MeanDifferenceVector.load(vector_path)
layer_ids = sorted(vector.layer_vectors.keys())
print('Examples:', len(dataset.examples), 'Layers:', layer_ids[:5], '...')

Examples: 60 Layers: [18, 19, 20, 21, 22] ...


In [3]:
model, tokenizer = load_model(model_name, device_map='auto', dtype='auto')

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [4]:
collector = ActivationCollector(
    model,
    tokenizer,
    layers=layer_ids,
    reduction='mean',
    response_only=True,
)
activations = collector.collect_dataset(dataset, progress=True, limit=1)  # Set to None to process full dataset

In [5]:
stats = compute_stats(activations, vector)
stats[:3]

[{'layer': 34,
  'mean_on': -155.4801788330078,
  'mean_off': -167.4405517578125,
  'diff': 11.960372924804688,
  'std_on': 1.960364390969097,
  'std_off': 2.331931669578779,
  'cohens_d': 5.552184293839461,
  'threshold': -161.46036529541016,
  'on_acc': 1.0,
  'off_acc': 1.0,
  'overall_acc': 1.0},
 {'layer': 33,
  'mean_on': -68.98149108886719,
  'mean_off': -77.91680908203125,
  'diff': 8.935317993164062,
  'std_on': 1.8263090194366636,
  'std_off': 1.6014448169177389,
  'cohens_d': 5.202330017634503,
  'threshold': -73.44915008544922,
  'on_acc': 1.0,
  'off_acc': 1.0,
  'overall_acc': 1.0},
 {'layer': 32,
  'mean_on': -75.67852020263672,
  'mean_off': -84.32289123535156,
  'diff': 8.644371032714844,
  'std_on': 2.163500551051331,
  'std_off': 1.9786819132057232,
  'cohens_d': 4.1696757946170475,
  'threshold': -80.00070571899414,
  'on_acc': 1.0,
  'off_acc': 1.0,
  'overall_acc': 1.0}]

In [9]:
import torch
top_layer = stats[0]['layer']
vec = vector.layer_vectors[top_layer].to(torch.float32)
vec /= torch.linalg.norm(vec) + 1e-8
proj_on = torch.stack(activations['on'][top_layer]).to(torch.float32) @ vec
proj_off = torch.stack(activations['off'][top_layer]).to(torch.float32) @ vec
proj_on, proj_off

(tensor([-154.4665, -159.3103, -153.8077, -154.7269, -155.0895]),
 tensor([-167.1023, -171.2284, -168.0110, -166.9004, -163.9606]))